In [ ]:
# pip install torch transformers sentence-transformers scikit-learn scipy openpyxl

In [ ]:
# SETUP: settings, output folders, settings check, input fingerprints
# Purpose: define every analysis setting in one place, create a new output folder for this run,
# and compare the settings with the ones saved on the first run.
# The data-collection files are only ever opened for reading. Their fingerprints (SHA-256) are
# taken here and checked again at the end (A9), which proves the analysis did not change them.
# Everything the analysis writes goes under analysis_results/, and each run gets its own new
# folder, so earlier results are never overwritten. Tables only: no charts are made.

import os, re, json, math, hashlib, datetime, itertools    # standard-library tools used throughout
from collections import Counter                            # counts labels and cluster sizes
import numpy as np                                         # numerical arrays
import pandas as pd                                        # tables

DEBUG = True                       # True = print extra [debug] lines; False = quieter output

def debug(*args):
    """Print a [debug] line, only when DEBUG is True."""
    if DEBUG:
        print("[debug]", *args)

# input locations (read-only)
INPUT_DIR = "question_samples_60"
PROMPTS_PATH = os.path.join(INPUT_DIR, "final_sample_60_prompts.csv")
RESPONSE_FILES = {"claude": "claude_responses/claude_outputs.jsonl",
                  "gemini": "gemini_responses/gemini_outputs.jsonl"}

# output locations
ANALYSIS_ROOT = "analysis_results"                         # everything the analysis writes lives under here
RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime("run_%Y%m%dT%H%M%SZ")
RUN_DIR = os.path.join(ANALYSIS_ROOT, RUN_ID)             # new folder for this run's results
CACHE_DIR = os.path.join(ANALYSIS_ROOT, "cache")          # NLI cache, shared by all runs (makes reruns fast)
VALIDATION_DIR = os.path.join(ANALYSIS_ROOT, "validation")  # reviewer sheets, shared by all runs
# the analysis folder must not be (or sit inside) a data-collection folder
protected_dirs = [os.path.abspath(INPUT_DIR)] + [os.path.abspath(os.path.dirname(p)) for p in RESPONSE_FILES.values()]
root_abs = os.path.abspath(ANALYSIS_ROOT)
for pdir in protected_dirs:
    assert os.path.commonpath([root_abs, pdir]) != pdir, f"{ANALYSIS_ROOT} must not be inside {pdir}"
for d in (RUN_DIR, CACHE_DIR, VALIDATION_DIR):
    os.makedirs(d, exist_ok=True)
print("this run writes to:", RUN_DIR)

# fingerprints of the data-collection files
def sha256(path):
    with open(path, "rb") as f:                            # "rb" = read-only
        return hashlib.sha256(f.read()).hexdigest()
INPUT_FILES = {"prompts": PROMPTS_PATH, **RESPONSE_FILES}
INPUT_HASHES = {name: sha256(p) for name, p in INPUT_FILES.items()}
for name, h in INPUT_HASHES.items():
    debug(f"input {name}: {INPUT_FILES[name]} sha256 {h[:12]}...")

# analysis settings
CONFIG = {
    "config_version": 1,                 # bump when you deliberately change a setting

    # validity and correctness
    "n_trials": 11,                      # trials per prompt
    "min_valid_trials": 9,               # a prompt needs at least 9 parseable trials to be included
    "valid_trial": "parsed Yes/No/Maybe label present (unparseable = missing)",
    "prompt_correctness": "majority label over valid trials == PubMedQA gold label; ties excluded (primary)",

    # semantic grouping (matches the pilot notebook, gemini_analysis.ipynb, method "label_nli_sbert")
    "response_text": "raw response with '*' removed, the Confidence line and everything after it removed, whitespace collapsed",
    "nli_model": "microsoft/deberta-large-mnli",
    "nli_include_context": True,             
    "nli_max_tokens": 512,                   # DeBERTa's limit; the abstract is shortened from the end to fit
    "nli_input": "premise = 'Context: <abstract> Question: <q> Answer: <response>', hypothesis = '<q> <other response>', both directions",
    "nli_equivalence": "strict",             # pilot: entailment must be the top label in BOTH directions
                                              # ('farquhar_default' = lenient alternative, not used)
    "nli_grouping": "greedy in trial order: a response joins the first group whose first member it is equivalent to",
    "sbert_model": "sentence-transformers/all-mpnet-base-v2",
    "sbert_text": "response text only (no question, no abstract), as in the pilot",
    "sbert_threshold": 0.85,                 # chosen from the pilot: at 0.90 paraphrases were still split apart
    "sbert_merge_rule": "NLI groups in order of first trial; each joins the first earlier merged group with the same label whose stored centroid has cosine >= threshold",
    "entropy_log_base": "e",                 # DSE in nats (natural log), over valid trials

    # evaluation
    "ece_bins": 10,                          # equal-width confidence bins for ECE
    "bootstrap_B": 2000,                     # bootstrap resamples
    "bootstrap_seed": 2026,                  # fixed seed so the intervals are reproducible
    "bootstrap_strata": "gold label (yes/no/maybe), resampling questions",
    "threshold_sensitivity": [0.80, 0.85, 0.90, 0.95],   # SBERT thresholds for the sensitivity analysis
    "validation_pairs_per_cell": 13,         # pairs per model x method x relation: 8 x 13 = 104 pairs per reviewer
    "validation_seed": 7,                    # fixed seed for drawing the validation pairs
}

# settings check (warns, never stops)
# The first run saves CONFIG as the reference. Later runs compare against it: any difference is
# printed and written to config_changes.jsonl, so it can be reported as a deviation.
reference_path = os.path.join(ANALYSIS_ROOT, "reference_config.json")
CONFIG_CHANGES = {}
if os.path.exists(reference_path):
    with open(reference_path) as f:
        reference = json.load(f)
    CONFIG_CHANGES = {k: {"reference": reference["config"].get(k), "this_run": v}
                      for k, v in CONFIG.items() if reference["config"].get(k) != v}
    if CONFIG_CHANGES:
        print(f"NOTE: {len(CONFIG_CHANGES)} setting(s) differ from the reference saved {reference['saved_utc']}:")
        for k, c in CONFIG_CHANGES.items():
            print(f"   {k}: {c['reference']} -> {c['this_run']}")
        print("   Logged in config_changes.jsonl - report it as a deviation if this run is used.")
        with open(os.path.join(ANALYSIS_ROOT, "config_changes.jsonl"), "a", encoding="utf-8") as f:
            f.write(json.dumps({"run_id": RUN_ID, "time_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
                                "changes": CONFIG_CHANGES}, default=str) + "\n")
    else:
        print(f"settings match the reference saved {reference['saved_utc']}")
else:
    with open(reference_path, "w") as f:
        json.dump({"saved_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(), "config": CONFIG}, f, indent=2)
    print("first run: settings saved as the reference ->", reference_path)

# names used by the later cells
EMOTIONS = {"N": "neutral", "F": "fear", "J": "joy"}     # template letter -> emotion
EMO_ORDER = ["all", "neutral", "fear", "joy"]
LABELS = ("yes", "no", "maybe")
# uncertainty measures compared by AUROC: display name -> column in the prompt-level table
MEASURES = {"VC": "mean_vc", "SC": "sc", "DSE": "dse_modified",
            "DSE_baseline": "dse_baseline", "LabelEntropy": "dse_label"}
# entropies are high when the model is uncertain, so they are negated before AUROC
# (every score then means "higher = more confident")
NEGATE = {"dse_modified", "dse_baseline", "dse_label"}

# every table for the final workbook is collected here: sheet name -> (description, DataFrame)
TABLES = {}
def add_table(name, df, description):
    """Register a table for the workbook (sheet names are limited to 31 characters)."""
    assert len(name) <= 31, f"sheet name too long: {name}"
    named_index = isinstance(df.index, pd.MultiIndex) or df.index.name is not None   # e.g. a groupby result
    TABLES[name] = (description, df.reset_index() if named_index else df.reset_index(drop=True))
    debug(f"table '{name}': {len(df)} rows")

add_table("config", pd.DataFrame([{"setting": k, "value": json.dumps(v) if isinstance(v, (list, dict)) else v,
                                   "changed_from_reference": k in CONFIG_CHANGES} for k, v in CONFIG.items()]),
          "Analysis settings used in this run, and whether each differs from the reference settings.")

In [ ]:
# LOAD AND PARSE THE RESPONSES (read-only on the response files)
# Purpose: read both models' raw outputs, pull the Yes/No/Maybe label, confidence and explanation
# out of each response, and mark which responses are valid (parseable).

def load_jsonl(path):
    """Read a JSONL file (one JSON record per line), skipping any broken line."""
    rows, bad = [], 0
    with open(path, "r", encoding="utf-8") as f:   # "r" = read-only; nothing is ever written back
        for line in f:
            if line.strip():                  # ignore blank lines
                try:
                    rows.append(json.loads(line))
                except json.JSONDecodeError:  # a line cut off by a crash
                    bad += 1
    if bad:
        print(f"WARNING: {bad} broken line(s) skipped in {path}")
    return rows

def parse_response(text):
    """Extract label, confidence and explanation from one raw response, and list any problems."""
    errors = []
    clean = re.sub(r"[*_`#]", "", str(text or ""))      # remove markdown symbols such as **Answer:**
    if not clean.strip():                               # nothing to parse
        return {"parsed_label": None, "confidence": np.nan, "explanation": None, "parse_errors": "empty_response"}

    # label: every "Answer: yes/no/maybe" in the text
    labels = [m.lower() for m in re.findall(r"answer\s*[:\-]\s*(yes|no|maybe)\b", clean, re.I)]
    label = labels[0] if labels else None
    if not labels:
        errors.append("no_answer_label")
    elif len(set(labels)) > 1:                          # e.g. "Answer: Yes ... Answer: No"
        errors.append("conflicting_answer_labels")
        label = None                                    # ambiguous -> treated as unparseable

    # explanation: the text after "Explanation:" up to the "Confidence:" line (or the end)
    m = re.search(r"explanation\s*[:\-]\s*(.*?)(?=\n\s*confidence\s*[:\-]|\Z)", clean, re.I | re.S)
    explanation = m.group(1).strip() if m and m.group(1).strip() else None
    if explanation is None:
        errors.append("no_explanation")

    # confidence: the number after "Confidence:", e.g. "85/100" or "85%"
    m = re.search(r"confidence\s*[:\-]\s*(\d{1,3}(?:\.\d+)?)\s*(?:/\s*100|%)?", clean, re.I)
    conf = float(m.group(1)) if m else np.nan
    if not m:
        errors.append("no_confidence")
    elif not 0 <= conf <= 100:                          # outside the allowed scale
        errors.append("confidence_out_of_range")
        conf = np.nan

    return {"parsed_label": label, "confidence": conf, "explanation": explanation,
            "parse_errors": ";".join(errors) or None}

# load both models into one table
calls = []
for model, path in RESPONSE_FILES.items():
    df = pd.DataFrame(load_jsonl(path))
    dupes = df.duplicated(["key", "trial"]).sum()       # should be 0 (collection never saves twice)
    if dupes:
        print(f"WARNING: {model}: {dupes} duplicate (key, trial) rows - keeping the first")
        df = df.drop_duplicates(["key", "trial"], keep="first")
    df["model"] = model
    calls.append(df)
calls = pd.concat(calls, ignore_index=True)

# parse every response and add the results as new columns
calls = pd.concat([calls, pd.DataFrame(calls["raw_response"].map(parse_response).tolist())], axis=1)
calls["valid"] = calls["parsed_label"].notna()          # valid = a usable Yes/No/Maybe label
# response-level correctness: 1/0 for valid responses, missing (NaN) for unparseable ones
calls["correct"] = np.where(calls["valid"], (calls["parsed_label"] == calls["ground_truth"]).astype(float), np.nan)
# response text for NLI and SBERT, exactly as in the pilot: '*' removed, the Confidence line (and anything
# after it) removed, whitespace collapsed
def response_text(text):
    t = str(text).replace("*", "")
    t = re.sub(r"\n?\s*confidence\s*:.*$", "", t, flags=re.I | re.S)
    return " ".join(t.split()).strip()
calls["resp_text"] = calls["raw_response"].map(response_text)
# each question's abstract, for the NLI premise (read from the prompt file, read-only)
context_of = pd.read_csv(PROMPTS_PATH, usecols=["id", "context"]).drop_duplicates("id").set_index("id")["context"]
calls["context"] = calls["question_id"].map(context_of)
assert calls["context"].notna().all(), "some questions have no abstract in the prompt file"
calls = calls.sort_values(["model", "question_id", "template_id", "trial"]).reset_index(drop=True)

# summary per model
for model, g in calls.groupby("model"):
    print(f"{model}: {len(g)} responses | {int((~g['valid']).sum())} unparseable | "
          f"{int(g['confidence'].isna().sum())} without confidence | parse errors: "
          f"{g['parse_errors'].dropna().str.split(';').explode().value_counts().to_dict()}")
    if DEBUG:                                           # show a few unparseable responses
        for _, r in g[~g["valid"]].head(3).iterrows():
            debug(f"  unparseable {r['key']} t{r['trial']}: {str(r['raw_response'])[:150]!r}")

In [ ]:
# SEMANTIC GROUPING: baseline (NLI only) and modified (label split + NLI + SBERT), as in the pilot
# Purpose: give every valid response a cluster ID under each method, using the same procedure as the
# pilot notebook (gemini_analysis.ipynb, "label_nli_sbert" method):
#   baseline  : NLI over all of a prompt's valid responses (no label split) - the original DSE procedure
#   modified  : split by Yes/No/Maybe -> NLI within each label -> SBERT merges NLI groups (same label only)
# Every NLI comparison (with its probabilities) and every SBERT comparison (with its cosine) is recorded,
# so each cluster assignment can be traced. NLI results are cached, so reruns and resumes are fast.

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer

# use a GPU if available (MPS on Apple Silicon, CUDA on NVIDIA), otherwise the CPU
device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# NLI model
nli_tok = AutoTokenizer.from_pretrained(CONFIG["nli_model"])
nli_model = AutoModelForSequenceClassification.from_pretrained(CONFIG["nli_model"]).to(device).eval()
ID2LABEL = {int(i): l.lower() for i, l in nli_model.config.id2label.items()}   # e.g. {0: 'contradiction', ...}
ENT = [i for i, l in ID2LABEL.items() if l == "entailment"][0]
print(f"NLI model: {CONFIG['nli_model']} | labels: {ID2LABEL}")

# SBERT model
sbert = SentenceTransformer(CONFIG["sbert_model"], device=device)
print(f"SBERT model: {CONFIG['sbert_model']} | threshold {CONFIG['sbert_threshold']}")

# NLI cache: probabilities for each (premise, hypothesis) pair already scored
NLI_CACHE_PATH = os.path.join(CACHE_DIR, f"nli_probs_{CONFIG['nli_model'].replace('/', '_')}.jsonl")
nli_cache = {}
if os.path.exists(NLI_CACHE_PATH):
    for row in load_jsonl(NLI_CACHE_PATH):
        nli_cache[row["k"]] = row["probs"]
debug(f"{len(nli_cache)} cached NLI pairs")

def pair_key(a, b):
    """A short fingerprint of an ordered (premise, hypothesis) pair, used as the cache key."""
    return hashlib.sha1((a + "\x1f" + b).encode("utf-8")).hexdigest()

def fit_context(ctx, q, a, b):
    """Shorten the abstract from the end so premise + hypothesis fit in nli_max_tokens (pilot rule).
    Returns (context to use, whether it was shortened)."""
    if not (CONFIG["nli_include_context"] and ctx):
        return "", False
    fixed = len(nli_tok(f"{q} {a}", f"{q} {b}")["input_ids"]) + 8
    room = CONFIG["nli_max_tokens"] - fixed
    ids = nli_tok(ctx, add_special_tokens=False)["input_ids"]
    if room <= 0:
        return "", True
    if len(ids) > room:
        return nli_tok.decode(ids[:room]), True
    return ctx, False

@torch.no_grad()
def nli_compare(q, ctx, a, b):
    """Compare responses a and b in both directions (pilot input format):
       premise    = 'Context: <abstract> Question: <q> Answer: <response>'
       hypothesis = '<q> <other response>'
    Returns probabilities, predicted labels and whether they count as equivalent."""
    c, shortened = fit_context(ctx, q, a, b)
    lead = f"Context: {c} Question: {q} Answer:" if c else f"{q}"
    pairs = [(f"{lead} {a}", f"{q} {b}"), (f"{lead} {b}", f"{q} {a}")]      # a -> b, then b -> a
    todo = [p for p in pairs if pair_key(*p) not in nli_cache]
    if todo:
        feats = nli_tok([p for p, _ in todo], [h for _, h in todo], padding=True, truncation=True,
                        max_length=CONFIG["nli_max_tokens"], return_tensors="pt").to(device)
        probs = torch.softmax(nli_model(**feats).logits, dim=-1).cpu().tolist()
        with open(NLI_CACHE_PATH, "a", encoding="utf-8") as f:              # save right away
            for p, pr in zip(todo, probs):
                k = pair_key(*p)
                nli_cache[k] = pr
                f.write(json.dumps({"k": k, "probs": pr}) + "\n")
    p_ab, p_ba = (nli_cache[pair_key(*p)] for p in pairs)
    pred_ab, pred_ba = int(np.argmax(p_ab)), int(np.argmax(p_ba))
    if CONFIG["nli_equivalence"] == "strict":            # pilot: entailment is the top label in both directions
        same = pred_ab == ENT and pred_ba == ENT
    else:                                                # Farquhar-style lenient rule
        labs = (ID2LABEL[pred_ab], ID2LABEL[pred_ba])
        same = "contradiction" not in labs and labs != ("neutral", "neutral")
    return {"p_ab": p_ab, "p_ba": p_ba, "pred_ab": ID2LABEL[pred_ab], "pred_ba": ID2LABEL[pred_ba],
            "equivalent": bool(same), "context_shortened": shortened}

NLI_LOG, SBERT_LOG = [], []          # every comparison made at the primary settings (become workbook sheets)

def nli_groups(items, q, ctx, method, model, key, log=True):
    """Greedy grouping (pilot rule): each response, in trial order, joins the first group whose FIRST
    member it is equivalent to; otherwise it starts a new group. items = list of (trial, text, label)."""
    groups = []
    for trial, text, lab in items:
        for g in groups:
            r = nli_compare(q, ctx, g[0][1], text)
            if log:
                NLI_LOG.append({"model": model, "key": key, "method": method,
                                "trial_a": g[0][0], "trial_b": trial, "label_a": g[0][2], "label_b": lab,
                                **{f"ab_{ID2LABEL[i]}": r["p_ab"][i] for i in ID2LABEL},
                                **{f"ba_{ID2LABEL[i]}": r["p_ba"][i] for i in ID2LABEL},
                                "pred_ab": r["pred_ab"], "pred_ba": r["pred_ba"],
                                "equivalent": r["equivalent"], "context_shortened": r["context_shortened"]})
            if r["equivalent"]:
                g.append((trial, text, lab))
                break
        else:
            groups.append([(trial, text, lab)])
    return [[t for t, _, _ in g] for g in groups]

def entropy(ids):
    """Discrete semantic entropy: -sum(p * ln p) over cluster shares among the valid trials."""
    if not len(ids):
        return np.nan
    p = np.array(list(Counter(ids).values()), dtype=float) / len(ids)
    return float(-(p * np.log(p)).sum()) + 0.0              # + 0.0 turns -0.0 into 0.0

# SBERT embeddings of every valid response (same text as the pilot: no question), computed once
valid = calls[calls["valid"]].copy()
print(f"embedding {len(valid)} valid responses with SBERT ...")
EMB = sbert.encode(valid["resp_text"].tolist(), batch_size=64, normalize_embeddings=True,
                   show_progress_bar=DEBUG, convert_to_numpy=True)
EMB_OF = dict(zip(valid.index, EMB))                         # response row -> embedding

def sbert_merge(nli_members, label_of, emb_of, threshold, model, key, log=True):
    """Pilot merge rule: NLI groups are taken in order of their first trial. Each joins the FIRST earlier
    merged group with the same label whose stored centroid (the centroid of that group's first NLI group)
    has cosine similarity >= threshold; otherwise it starts a new merged group. Can only merge, never split."""
    def centroid(ts):
        v = np.mean([emb_of[t] for t in ts], axis=0)
        return v / np.linalg.norm(v)
    merged = []                                              # each: [label, centroid, trials]
    for ts in sorted(nli_members, key=min):
        lab, c = label_of[ts[0]], centroid(ts)
        for m in merged:
            if m[0] != lab:
                continue                                     # different labels never merge
            cos = float(np.dot(m[1], c))
            hit = cos >= threshold
            if log:
                SBERT_LOG.append({"model": model, "key": key, "label": lab, "threshold": threshold,
                                  "group_trials": json.dumps(sorted(ts)), "candidate_trials": json.dumps(sorted(m[2])),
                                  "cosine": cos, "merged": hit})
            if hit:
                m[2].extend(ts)
                break
        else:
            merged.append([lab, c, list(ts)])
    return [m[2] for m in merged]

def group_prompt(g, threshold, log=True):
    """Cluster one prompt's valid trials. Returns {response row: id} for baseline, NLI-within-label and modified."""
    model, key = g["model"].iloc[0], g["key"].iloc[0]
    q, ctx = str(g["question"].iloc[0]), str(g["context"].iloc[0] or "")
    rows = list(g.index)
    trial_row = dict(zip(g["trial"], rows))
    items = [(int(t), x, l) for t, x, l in zip(g["trial"], g["resp_text"], g["parsed_label"])]
    # baseline: NLI on all responses together
    base = nli_groups(items, q, ctx, "baseline", model, key, log)
    # modified: label split, NLI within each label, then SBERT merge
    within = []
    for lab in dict.fromkeys(l for _, _, l in items):
        within += nli_groups([it for it in items if it[2] == lab], q, ctx, "within_label", model, key, log)
    label_of = {t: l for t, _, l in items}
    emb_of = {t: EMB_OF[trial_row[t]] for t in label_of}
    modified = sbert_merge(within, label_of, emb_of, threshold, model, key, log)
    to_ids = lambda groups, tag: {trial_row[t]: f"{tag}{gi}" for gi, ts in enumerate(groups) for t in ts}
    return to_ids(base, "b"), to_ids(within, "w"), to_ids(modified, "m")

# cluster every prompt of both models
t0 = datetime.datetime.now()
ids = {"cluster_baseline": {}, "cluster_nli_within_label": {}, "cluster_modified": {}}
groups = list(valid.groupby(["model", "key"], sort=False))
for n, ((model, key), g) in enumerate(groups, 1):
    for col, part in zip(ids, group_prompt(g.sort_values("trial"), CONFIG["sbert_threshold"])):
        ids[col].update(part)
    if n % 50 == 0 or n == len(groups):                     # progress line
        print(f"grouped {n}/{len(groups)} prompts | {len(nli_cache)} NLI pairs cached | "
              f"{(datetime.datetime.now() - t0).seconds // 60} min")

# attach the cluster IDs to the responses (cluster IDs are prefixed by label so they never span labels)
for col, d in ids.items():
    calls[col] = pd.Series(d)
for col in ("cluster_nli_within_label", "cluster_modified"):
    calls[col] = np.where(calls[col].notna(), calls["parsed_label"] + "-" + calls[col].astype(str), None)
calls["cluster_label"] = calls["parsed_label"]              # label-only "clusters" for label entropy

add_table("nli_pairs", pd.DataFrame(NLI_LOG),
          "Every NLI comparison made: the two trials, NLI probabilities in both directions (a->b, b->a), "
          "predicted labels, whether they counted as equivalent, and whether the abstract was shortened.")
add_table("sbert_merges", pd.DataFrame(SBERT_LOG),
          "Every SBERT comparison in the modified method (same label only): the NLI group being placed, the "
          "candidate merged group, their centroid cosine similarity, and whether they merged.")
print(f"grouping done | {len(NLI_LOG)} NLI comparisons | {len(SBERT_LOG)} SBERT comparisons | "
      f"abstract shortened in {sum(r['context_shortened'] for r in NLI_LOG)} comparisons")

In [ ]:
# PROMPT-LEVEL TABLE AND DATA-QUALITY TABLES
# Purpose: one row per model x prompt with everything later cells need: validity, majority
# answer, ties, correctness, SC, VC and the three entropies.

def prompt_row(g):
    """Summarise one prompt's trials (g = all trials of one model x prompt)."""
    f = g.iloc[0]                                           # shared fields (question, gold, emotion, ...)
    v = g[g["valid"]]                                       # valid trials only
    n = len(v)
    counts = Counter(v["parsed_label"])                     # e.g. {'yes': 8, 'maybe': 3}
    top = max(counts.values()) if n else 0                  # size of the most common label
    tied = sorted(l for l, c in counts.items() if c == top) if n else []
    majority = tied[0] if len(tied) == 1 else None          # no majority on a tie
    return {
        # identifiers
        "model": f["model"], "key": f["key"], "question_id": f["question_id"], "pub_id": f["pub_id"],
        "question": f["question"], "gold": f["ground_truth"], "emotion": f["emotion"], "template_id": f["template_id"],
        # validity
        "n_trials": len(g), "n_valid": n, "n_unparseable": len(g) - n,
        "included": n >= CONFIG["min_valid_trials"],
        # answers
        **{f"count_{l}": counts.get(l, 0) for l in LABELS},
        "majority": majority, "tie": len(tied) > 1, "tied_labels": "|".join(tied) or None,
        # prompt-level correctness
        "correct": float(majority == f["ground_truth"]) if majority else np.nan,         # primary: ties missing
        "correct_tie_wrong": float(majority == f["ground_truth"]) if majority else 0.0,  # sensitivity: ties wrong
        # uncertainty measures
        "sc": top / n if n else np.nan,                     # SC Agreement over valid trials only
        "mean_vc": v["confidence"].mean() / 100 if v["confidence"].notna().any() else np.nan,   # VC on a 0-1 scale
        "sd_vc": v["confidence"].std() / 100 if v["confidence"].notna().sum() > 1 else np.nan,
        "n_resp_correct": float(v["correct"].sum()),        # correct valid responses (response-level accuracy)
        "dse_label": entropy(v["cluster_label"].tolist()),        # label entropy (reference)
        "dse_baseline": entropy(v["cluster_baseline"].tolist()),  # DSE, baseline grouping
        "dse_modified": entropy(v["cluster_modified"].tolist()),  # DSE, modified grouping (primary)
        # grouping diagnostics
        "n_clusters_label": v["cluster_label"].nunique(),
        "n_clusters_baseline": v["cluster_baseline"].nunique(),
        "n_clusters_modified": v["cluster_modified"].nunique(),
        # baseline clusters that contain more than one answer label (cross-label merges)
        "baseline_mixed_labels": int(v.groupby("cluster_baseline")["parsed_label"].nunique().gt(1).sum()),
        "thinking_tokens_mean": pd.to_numeric(g.get("tok_thinking_tokens"), errors="coerce").mean()
                                if "tok_thinking_tokens" in g else np.nan,
    }

prompts_df = pd.DataFrame([prompt_row(g) for _, g in calls.groupby(["model", "key"], sort=False)])
inc = prompts_df[prompts_df["included"]]                    # prompts with >= 9 valid trials

# data-quality table
dq = []
for model, g in calls.groupby("model"):
    p = prompts_df[prompts_df["model"] == model]
    errs = g["parse_errors"].dropna().str.split(";").explode().value_counts().to_dict()
    dq.append({"model": model, "responses": len(g), "valid_responses": int(g["valid"].sum()),
               "unparseable_responses": int((~g["valid"]).sum()),
               "valid_without_confidence": int((g["valid"] & g["confidence"].isna()).sum()),
               **{f"parse_error_{k}": v for k, v in errs.items()},
               "prompts": len(p), "prompts_excluded_lt_min_valid": int((~p["included"]).sum()),
               "prompts_included": int(p["included"].sum()),
               "ties_among_included": int(p.loc[p["included"], "tie"].sum()),
               "model_versions": "|".join(sorted(g["model_version"].astype(str).unique())) if "model_version" in g else None})
dq = pd.DataFrame(dq)
print(dq.T.to_string())
add_table("data_quality", dq, "Responses, unparseable responses, parse errors, excluded prompts and ties per model.")

# ties: which prompts tied and how
ties = inc[inc["tie"]][["model", "key", "question_id", "gold", "emotion", "template_id", "n_valid",
                        "count_yes", "count_no", "count_maybe", "tied_labels"]]
add_table("ties", ties, "Included prompts whose valid trials had no single majority label (excluded from the primary analysis).")

In [ ]:
# METRICS AND STRATIFIED QUESTION-LEVEL BOOTSTRAP
# Purpose: RQ1 accuracy, RQ2/RQ3 calibration (Brier, ECE) and discrimination (AUROC),
# robustness (change in AUROC vs neutral), each with a 95% bootstrap interval,
# the tie sensitivity analysis, and Claude-minus-Gemini differences.

from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr

def ece(conf, y, bins):
    """Expected Calibration Error: the weighted average gap between confidence and accuracy
    across equal-width confidence bins."""
    conf, y = np.asarray(conf, float), np.asarray(y, float)
    edges = np.linspace(0, 1, bins + 1)
    idx = np.clip(np.digitize(conf, edges[1:-1], right=True), 0, bins - 1)    # bin of each prompt
    return float(sum(abs(y[idx == b].mean() - conf[idx == b].mean()) * (idx == b).mean()
                     for b in range(bins) if (idx == b).any()))

def auroc(score, y):
    """AUROC of a confidence score for predicting correctness; NaN if only one class is present."""
    y = np.asarray(y)
    return roc_auc_score(y, score) if len(set(y)) == 2 else np.nan

def stats(d, correct_col="correct"):
    """Every scalar result for one set of included prompts of one model.
    correct_col = 'correct' (ties excluded) or 'correct_tie_wrong' (ties counted as wrong)."""
    out = {}
    for emo in EMO_ORDER:
        e = d if emo == "all" else d[d["emotion"] == emo]
        # RQ1, response level: correct valid responses / all valid responses
        out[f"acc_response_{emo}"] = e["n_resp_correct"].sum() / e["n_valid"].sum() if len(e) else np.nan
        # prompts with a correctness value and a VC (ties drop out in the primary analysis)
        s = e[e[correct_col].notna() & e["mean_vc"].notna()]
        y = s[correct_col].values
        out[f"acc_prompt_{emo}"] = y.mean() if len(y) else np.nan                          # RQ1, prompt level
        out[f"brier_vc_{emo}"] = float(np.mean((s["mean_vc"] - y) ** 2)) if len(y) else np.nan   # Brier of VC
        out[f"ece_vc_{emo}"] = ece(s["mean_vc"], y, CONFIG["ece_bins"]) if len(y) else np.nan   # ECE of VC
        for name, col in MEASURES.items():                                                  # AUROC per measure
            sc = -s[col] if col in NEGATE else s[col]
            out[f"auroc_{name}_{emo}"] = auroc(sc, y) if len(y) else np.nan
    # robustness: change in AUROC from neutral to fear and to joy
    for name in MEASURES:
        for emo in ["fear", "joy"]:
            out[f"dauroc_{name}_{emo}"] = out[f"auroc_{name}_{emo}"] - out[f"auroc_{name}_neutral"]
    # how much DSE adds beyond label consistency, and the modified vs baseline grouping
    out["auroc_DSE_minus_LabelEntropy_all"] = out["auroc_DSE_all"] - out["auroc_LabelEntropy_all"]
    out["auroc_DSE_minus_baseline_all"] = out["auroc_DSE_all"] - out["auroc_DSE_baseline_all"]
    return out

# the 60 questions per gold class; both models resample from the same lists with the same seed,
# so draw b uses the same questions for Claude and Gemini (needed for the model differences)
ALL_QS = prompts_df.groupby("gold")["question_id"].unique()

def bootstrap(d, stat_fn, strata=None, B=CONFIG["bootstrap_B"], seed=CONFIG["bootstrap_seed"]):
    """Point estimate plus percentile 95% CI for every result of stat_fn.
    Resamples questions (with all their prompts) with replacement, within each gold class,
    so every resample keeps the class balance. Returns (summary table, all draws)."""
    strata = ALL_QS if strata is None else strata
    rng = np.random.default_rng(seed)
    by_q = {q: g for q, g in d.groupby("question_id")}      # each question's prompts
    point = stat_fn(d)                                      # estimate on the real data
    draws = []
    for b in range(B):
        sample = [q for lab in strata.index for q in rng.choice(strata[lab], size=len(strata[lab]), replace=True)]
        draws.append(stat_fn(pd.concat([by_q[q] for q in sample if q in by_q], ignore_index=True)))
        if DEBUG and b in (0, B // 2):
            debug(f"bootstrap draw {b + 1}/{B}")
    draws = pd.DataFrame(draws)
    summary = pd.DataFrame({"estimate": pd.Series(point),
                            "ci_low": draws.quantile(0.025), "ci_high": draws.quantile(0.975),
                            "n_valid_draws": draws.notna().sum()})   # draws where the metric was defined
    return summary, draws

# primary analysis and tie sensitivity analysis, per model
results, DRAWS = {}, {}
for model, d in inc.groupby("model"):
    for analysis, col in [("primary", "correct"), ("ties_wrong", "correct_tie_wrong")]:
        print(f"bootstrapping {model} ({analysis}) ...")
        results[(model, analysis)], DRAWS[(model, analysis)] = bootstrap(d, lambda x, c=col: stats(x, c))

main = pd.concat(results, names=["model", "analysis", "metric"]).reset_index()

# Claude minus Gemini, from paired draws
models = sorted(inc["model"].unique())
if len(models) == 2:
    a, b = models
    pt = results[(a, "primary")]["estimate"] - results[(b, "primary")]["estimate"]
    dd = DRAWS[(a, "primary")] - DRAWS[(b, "primary")]
    model_diff = pd.DataFrame({"comparison": f"{a} - {b}", "estimate": pt,
                               "ci_low": dd.quantile(0.025), "ci_high": dd.quantile(0.975)})
    model_diff.index.name = "metric"
    model_diff = model_diff.reset_index()

def split_metric(df):
    """Split metric names like 'auroc_VC_fear' into columns: metric family, measure, emotion."""
    df = df.copy()
    df["emotion"] = df["metric"].str.rsplit("_", n=1).str[-1]
    df["family"] = df["metric"].str.split("_").str[0]
    df["measure"] = df["metric"].str.split("_", n=1).str[1].str.rsplit("_", n=1).str[0]
    df["formatted"] = [f"{e:.3f} [{l:.3f}, {h:.3f}]" if pd.notna(e) else ""
                       for e, l, h in zip(df["estimate"], df["ci_low"], df["ci_high"])]
    return df

main = split_metric(main)
prim = main[main["analysis"] == "primary"]

def wide(df, families, index=("family", "measure", "emotion")):
    """Table with one row per metric and estimate / CI columns per model."""
    t = df[df["family"].isin(families)]
    out = t.pivot_table(index=list(index), columns="model",
                        values=["estimate", "ci_low", "ci_high", "formatted"], aggfunc="first")
    out.columns = [f"{m}_{v}" for v, m in out.columns]
    ordered = [f"{m}_{v}" for m in sorted(df["model"].unique()) for v in ("estimate", "ci_low", "ci_high", "formatted")]
    out = out[[c for c in ordered if c in out.columns]].reset_index()
    order = {e: i for i, e in enumerate(EMO_ORDER + ["fear", "joy"])}
    return out.sort_values(list(index[:-1]) + ["emotion"], key=lambda s: s.map(order) if s.name == "emotion" else s)

add_table("rq1_accuracy", wide(prim, ["acc"]),
          "RQ1: response-level accuracy (correct valid responses / valid responses) and prompt-level accuracy "
          "(majority label == gold; ties excluded), overall and by emotion, with 95% stratified bootstrap CIs.")
add_table("calibration_vc", wide(prim, ["brier", "ece"]),
          "RQ2/RQ3: Brier score and ECE (10 bins) of prompt-level mean verbalized confidence against prompt-level correctness.")
add_table("auroc", wide(prim, ["auroc"]),
          "RQ2/RQ3: AUROC of each uncertainty measure for predicting prompt-level correctness (entropies negated). "
          "Includes DSE minus label entropy and DSE minus baseline DSE.")
add_table("robustness_dauroc", wide(prim, ["dauroc"]),
          "Robustness: change in AUROC from neutral to fear and to joy, per measure.")
add_table("tie_sensitivity", wide(main[main["analysis"] == "ties_wrong"], ["acc", "brier", "ece", "auroc"]),
          "Sensitivity analysis: all metrics recomputed with tied prompts counted as incorrect.")
if len(models) == 2:
    add_table("model_difference", split_metric(model_diff.assign(model=model_diff["comparison"])).drop(columns="model"),
              f"Difference {a} minus {b} for every metric, with paired 95% bootstrap CIs (same questions in each draw).")
add_table("all_metrics_long", main,
          "Every metric for both models and both analyses (primary, ties_wrong) in long format.")

print("\n=== RQ1 accuracy ===");    print(TABLES["rq1_accuracy"][1].filter(regex="family|measure|emotion|formatted").to_string(index=False))
print("\n=== AUROC (all) ===");     print(TABLES["auroc"][1].query("emotion == 'all'").filter(regex="measure|formatted").to_string(index=False))

In [ ]:
# BREAKDOWNS: GOLD CLASS, TEMPLATE, CONFUSION, DESCRIPTIVES, CORRELATIONS, GROUPING
# Purpose: the remaining tables for the results and discussion.

# by gold class (bootstrap within the class)
by_class = {}
for (model, gold), d in inc.groupby(["model", "gold"]):
    print(f"bootstrapping {model} / gold={gold} ...")
    by_class[(model, gold)], _ = bootstrap(d, lambda x: stats(x, "correct"), strata=ALL_QS[[gold]])
by_class = split_metric(pd.concat(by_class, names=["model", "gold", "metric"]).reset_index())
add_table("by_gold_class", by_class.sort_values(["model", "gold", "family", "measure", "emotion"]),
          "Every metric computed separately for gold yes, no and maybe questions (20 questions each), "
          "with 95% bootstrap CIs resampling questions within the class.")

# by template (point estimates; each template has 60 prompts)
tmpl_rows = []
for (model, t), d in inc.groupby(["model", "template_id"]):
    s = stats(d)
    tmpl_rows.append({"model": model, "template_id": t, "emotion": EMOTIONS[t[0]],
                      **{k[:-4]: v for k, v in s.items() if k.endswith("_all") and not k.startswith("dauroc")}})
add_table("by_template", pd.DataFrame(tmpl_rows),
          "Accuracy, VC calibration and AUROC per template (point estimates; 60 prompts per template).")

# majority answer vs gold label
conf_rows = []
for (model, emo), d in pd.concat([inc.assign(emotion="all"), inc]).groupby(["model", "emotion"]):
    m = d.assign(majority=d["majority"].fillna("TIE"))
    ct = pd.crosstab(m["gold"], m["majority"]).reindex(index=list(LABELS), columns=list(LABELS) + ["TIE"], fill_value=0)
    for gold, row in ct.iterrows():
        conf_rows.append({"model": model, "emotion": emo, "gold": gold, **{f"majority_{c}": int(v) for c, v in row.items()}})
add_table("answer_confusion", pd.DataFrame(conf_rows),
          "Counts of included prompts by gold label (rows) and majority answer (columns), overall and by emotion.")

# descriptive statistics of the uncertainty measures
desc_rows = []
for (model, emo), d in pd.concat([inc.assign(emotion="all"), inc]).groupby(["model", "emotion"]):
    for name, col in {"VC": "mean_vc", "SC": "sc", "DSE_modified": "dse_modified",
                      "DSE_baseline": "dse_baseline", "LabelEntropy": "dse_label"}.items():
        x = d[col].dropna()
        desc_rows.append({"model": model, "emotion": emo, "measure": name, "n_prompts": len(x),
                          "mean": x.mean(), "sd": x.std(), "median": x.median(),
                          "q25": x.quantile(0.25), "q75": x.quantile(0.75), "min": x.min(), "max": x.max(),
                          "mean_if_correct": d.loc[d["correct"] == 1, col].mean(),
                          "mean_if_incorrect": d.loc[d["correct"] == 0, col].mean()})
add_table("measure_descriptives", pd.DataFrame(desc_rows),
          "Distribution of VC, SC, DSE (modified and baseline) and label entropy per model and emotion, "
          "including means for correct and incorrect prompts.")

# Spearman correlations between the uncertainty measures
corr_rows = []
cols = ["mean_vc", "sc", "dse_modified", "dse_baseline", "dse_label"]
for (model, emo), d in pd.concat([inc.assign(emotion="all"), inc]).groupby(["model", "emotion"]):
    for x, y in itertools.combinations(cols, 2):             # every pair of measures
        s = d[[x, y]].dropna()
        rho, p = spearmanr(s[x], s[y]) if len(s) > 2 else (np.nan, np.nan)
        corr_rows.append({"model": model, "emotion": emo, "measure_a": x, "measure_b": y,
                          "spearman_rho": rho, "p_value": p, "n_prompts": len(s)})
add_table("correlations", pd.DataFrame(corr_rows),
          "Spearman correlations between VC, SC, DSE (modified, baseline) and label entropy, per model and emotion.")

# how much DSE adds beyond the Yes/No/Maybe label
extra_rows = []
for (model, emo), d in pd.concat([inc.assign(emotion="all"), inc]).groupby(["model", "emotion"]):
    more = d["dse_modified"] > d["dse_label"] + 1e-12           # DSE sees splits within a label
    extra_rows.append({"model": model, "emotion": emo, "n_prompts": len(d),
                       "share_dse_above_label_entropy": more.mean(),
                       "share_sc_1_but_dse_above_0": ((d["sc"] == 1) & (d["dse_modified"] > 1e-12)).mean(),
                       "mean_dse_minus_label_entropy": (d["dse_modified"] - d["dse_label"]).mean()})
add_table("dse_beyond_label", pd.DataFrame(extra_rows),
          "How often DSE differs from plain label entropy (i.e. responses with the same label were split by meaning).")

# baseline vs modified grouping
cmp_ = inc.groupby("model").agg(
    prompts=("key", "size"),
    mean_clusters_label=("n_clusters_label", "mean"),
    mean_clusters_baseline=("n_clusters_baseline", "mean"),
    mean_clusters_modified=("n_clusters_modified", "mean"),
    mean_dse_label=("dse_label", "mean"),
    mean_dse_baseline=("dse_baseline", "mean"),
    mean_dse_modified=("dse_modified", "mean"),
    prompts_with_mixed_label_baseline_cluster=("baseline_mixed_labels", lambda s: int((s > 0).sum())))
add_table("grouping_comparison", cmp_,
          "Baseline (original DSE clustering) vs modified (label split + NLI + SBERT) grouping: clusters, "
          "entropy, and prompts where a baseline cluster mixed answer labels.")

# Gemini's occasional thinking (responses with vs without thinking tokens)
if "tok_thinking_tokens" in calls:
    t = calls[calls["valid"]].assign(thinking=pd.to_numeric(calls["tok_thinking_tokens"], errors="coerce"))
    t = t[t["thinking"].notna()]
    if len(t):
        add_table("thinking_tokens",
                  t.assign(thought=t["thinking"] > 0).groupby(["model", "emotion", "thought"]).agg(
                      responses=("key", "size"), accuracy=("correct", "mean"),
                      mean_confidence=("confidence", "mean"), mean_thinking_tokens=("thinking", "mean")),
                  "Responses with and without reasoning tokens (Gemini): count, accuracy and mean stated confidence.")
print("breakdown tables done")

In [ ]:
# THRESHOLD SENSITIVITY (reported separately; the primary threshold is unchanged)
# Purpose: show how DSE and its AUROC change with the SBERT threshold.
# NLI probabilities come from the cache, so only the SBERT merge step is really redone.

sens = []
for thr in CONFIG["threshold_sensitivity"]:
    mod = {}
    for (model, key), g in valid.groupby(["model", "key"], sort=False):
        _, _, m = group_prompt(g.sort_values("trial"), thr, log=False)   # re-cluster with this threshold
        mod[(model, key)] = (entropy(list(m.values())), len(set(m.values())))
    d = inc.copy()
    d["dse_modified"] = [mod.get((m, k), (np.nan, np.nan))[0] for m, k in zip(d["model"], d["key"])]
    d["n_clusters_modified"] = [mod.get((m, k), (np.nan, np.nan))[1] for m, k in zip(d["model"], d["key"])]
    for model, dm in d.groupby("model"):
        s = stats(dm)
        sens.append({"threshold": thr, "model": model, "primary": thr == CONFIG["sbert_threshold"],
                     "mean_clusters": dm["n_clusters_modified"].mean(), "mean_dse": dm["dse_modified"].mean(),
                     **{f"auroc_DSE_{e}": s[f"auroc_DSE_{e}"] for e in EMO_ORDER},
                     **{f"dauroc_DSE_{e}": s[f"dauroc_DSE_{e}"] for e in ("fear", "joy")}})
add_table("threshold_sensitivity", pd.DataFrame(sens),
          "DSE (modified) recomputed at each SBERT threshold: clusters, mean DSE, AUROC and change in AUROC. "
          "The primary threshold is marked.")
print(pd.DataFrame(sens).round(3).to_string(index=False))

In [ ]:
# MANUAL VALIDATION OF SEMANTIC GROUPING: export blinded sheets for two reviewers
# Pairs of responses from the same prompt, sampled per model x method x relation:
#   same_cluster -> checks over-merging (grouped together but different meaning); any labels
#   diff_cluster -> checks over-splitting (kept apart but same meaning); same Yes/No/Maybe label only,
#                   since responses with different labels cannot have the same meaning
# Reviewers see only the two responses; method and relation stay in the separate key file.
# The sheets go to VALIDATION_DIR (shared by all runs) and are never overwritten.

key_path = os.path.join(VALIDATION_DIR, "validation_key.csv")     # unblinded key: do not give to reviewers
if os.path.exists(key_path):
    print(f"{key_path} already exists - not overwritten (reviewers may have started)")
else:
    rng = np.random.default_rng(CONFIG["validation_seed"])
    v = calls[calls["valid"]]
    pairs = []
    for model in v["model"].unique():
        vm = v[v["model"] == model]
        for method, col in [("baseline", "cluster_baseline"), ("modified", "cluster_modified")]:
            cand = {"same_cluster": [], "diff_cluster": []}          # candidate pairs per relation
            for _, g in vm.groupby("key"):
                for i, j in itertools.combinations(g.index, 2):     # every pair within one prompt
                    if g.at[i, "raw_response"] == g.at[j, "raw_response"]:
                        continue                                     # identical text tells the reviewers nothing
                    rel = "same_cluster" if g.at[i, col] == g.at[j, col] else "diff_cluster"
                    if rel == "diff_cluster" and g.at[i, "parsed_label"] != g.at[j, "parsed_label"]:
                        continue
                    cand[rel].append((i, j))
            for rel, lst in cand.items():                            # random sample from each group
                k = min(CONFIG["validation_pairs_per_cell"], len(lst))
                for n in rng.choice(len(lst), size=k, replace=False) if k else []:
                    i, j = lst[n]
                    pairs.append({"model": model, "method": method, "relation": rel, "key": v.at[i, "key"],
                                  "response_a": v.at[i, "raw_response"], "response_b": v.at[j, "raw_response"]})
    # shuffle so reviewers cannot tell method or relation from the order
    pairs = pd.DataFrame(pairs).sample(frac=1, random_state=CONFIG["validation_seed"]).reset_index(drop=True)
    pairs.insert(0, "pair_id", range(1, len(pairs) + 1))
    pairs.to_csv(key_path, index=False)
    for reviewer in ("reviewer1", "reviewer2"):                 # one blinded sheet per reviewer
        sheet = pairs[["pair_id", "response_a", "response_b"]].assign(same_meaning="", notes="")
        sheet.to_csv(os.path.join(VALIDATION_DIR, f"validation_{reviewer}.csv"), index=False, encoding="utf-8-sig")
    print(f"{len(pairs)} pairs exported to {VALIDATION_DIR}:",
          pairs.groupby(["model", "method", "relation"]).size().to_dict())

print("""Reviewer instructions: for each pair, fill same_meaning with Y or N, independently.
Y = same final answer AND the same main evidence/reasoning (wording, order and level of detail may differ).
N = a different answer, or a different main reason/evidence, or one adds/contradicts a key claim.
Over-merge = same_cluster pair judged N. Over-split = diff_cluster pair judged Y.""")

In [ ]:
# VALIDATION RESULTS: agreement between reviewers, then error rates from consensus labels
# Reads the reviewer sheets as .xlsx or .csv. If validation_consensus (.csv/.xlsx) exists, error rates use
# the consensus labels for all pairs; otherwise they use only the pairs both reviewers agreed on.

from sklearn.metrics import cohen_kappa_score

def read_sheet(name):
    """Read analysis_results/validation/<name>.xlsx or .csv, whichever exists (None if neither)."""
    for ext, reader in ((".xlsx", pd.read_excel), (".csv", pd.read_csv)):
        p = os.path.join(VALIDATION_DIR, name + ext)
        if os.path.exists(p):
            return reader(p)
    return None

def yn(s):
    """'yes', 'Y ', 'y' -> 'Y'; anything else -> missing."""
    s = s.astype(str).str.strip().str.upper().str[0]
    return s.where(s.isin(["Y", "N"]))

key = pd.read_csv(key_path)
r1, r2, cons = read_sheet("validation_reviewer1"), read_sheet("validation_reviewer2"), read_sheet("validation_consensus")

if r1 is None or r2 is None:
    print("reviewer sheets not found - validation tables skipped for this run")
else:
    m = (key.merge(r1[["pair_id", "same_meaning"]].rename(columns={"same_meaning": "r1"}), on="pair_id")
            .merge(r2[["pair_id", "same_meaning"]].rename(columns={"same_meaning": "r2"}), on="pair_id"))
    m["r1"], m["r2"] = yn(m["r1"]), yn(m["r2"])
    done = m.dropna(subset=["r1", "r2"])
    agree = float((done["r1"] == done["r2"]).mean())
    kappa = cohen_kappa_score(done["r1"], done["r2"])
    print(f"{len(done)}/{len(m)} pairs rated by both | agreement {agree:.3f} | Cohen's kappa {kappa:.3f}")

    # final label per pair: the consensus if available, otherwise the shared answer where the reviewers agreed
    m["final"] = np.where(m["r1"] == m["r2"], m["r1"], None)
    if cons is not None:
        c = cons.set_index("pair_id")["consensus"]
        m["final"] = yn(m["pair_id"].map(c)).fillna(m["final"])
        label_source = "consensus after discussion"
    else:
        label_source = "pairs both reviewers agreed on (no consensus file yet)"
    unresolved = int(m["final"].isna().sum())
    print(f"error rates use: {label_source} | unresolved pairs: {unresolved}")

    add_table("validation_agreement", pd.DataFrame([{
        "pairs_exported": len(m), "pairs_rated_by_both": len(done), "percent_agreement": agree,
        "cohens_kappa": kappa, "disagreements": int((done["r1"] != done["r2"]).sum()),
        "reviewer1_Y": int((m["r1"] == "Y").sum()), "reviewer2_Y": int((m["r2"] == "Y").sum()),
        "error_rates_based_on": label_source, "unresolved_pairs": unresolved}]),
        "Initial agreement between the two independent reviewers, and which labels the error rates use.")

    # compare the final human label with what the algorithm did (from the key)
    f = m.dropna(subset=["final"]).copy()
    f["error"] = np.where(f["relation"] == "same_cluster", f["final"] == "N",    # grouped but different: over-merge
                                                            f["final"] == "Y")    # separated but same: over-split
    val = f.groupby(["model", "method", "relation"]).agg(pairs=("error", "size"), errors=("error", "sum"),
                                                         error_rate=("error", "mean"))
    val = val.rename(index={"same_cluster": "over-merge", "diff_cluster": "over-split"}, level="relation")
    add_table("validation_error_rates", val, f"Over-merge and over-split rates per model and method ({label_source}).")
    add_table("validation_pairs", m.drop(columns=["response_a", "response_b"]),
              "Every validation pair with both reviewers' ratings and the final label used.")
    print(val.round(3).to_string())

In [ ]:
# CHECK THE INPUTS AND WRITE EVERYTHING TO ONE EXCEL WORKBOOK
# Purpose: confirm the data-collection files are byte-for-byte unchanged, then save one file with
# every table plus the prompt-level and response-level data, in this run's own folder.

add_table("prompt_level_data", prompts_df,
          "One row per model x prompt: validity, majority answer, tie, correctness, SC, VC, DSE "
          "(label, baseline, modified) and cluster counts.")
resp = calls.drop(columns=["resp_text", "context"], errors="ignore").copy()
for c in resp.columns:                                   # nested values (e.g. settings) -> text
    if resp[c].map(lambda x: isinstance(x, (dict, list))).any():
        resp[c] = resp[c].map(lambda x: json.dumps(x, default=str) if isinstance(x, (dict, list)) else x)
add_table("response_level_data", resp,
          "One row per response: raw response, parsed label, confidence, explanation, parse errors, "
          "correctness and cluster IDs under each grouping method.")

# prove the data-collection files were not changed by the analysis
after = {name: sha256(p) for name, p in INPUT_FILES.items()}
changed_inputs = [n for n in INPUT_HASHES if INPUT_HASHES[n] != after[n]]
assert not changed_inputs, f"input files changed during the analysis: {changed_inputs}"
print("data-collection files unchanged (SHA-256 identical before and after)")
add_table("inputs", pd.DataFrame([{"input": n, "path": INPUT_FILES[n], "sha256": INPUT_HASHES[n],
                                   "unchanged_after_analysis": True} for n in INPUT_FILES]),
          "The data-collection files this run read, with SHA-256 fingerprints taken before and after the analysis.")

# software environment of this analysis run
import sys, platform, importlib.metadata
env = {"python": sys.version.split()[0], "platform": platform.platform(), "device": device}
for pkg in ["torch", "transformers", "sentence-transformers", "scikit-learn", "scipy", "numpy", "pandas", "openpyxl"]:
    try:
        env[pkg] = importlib.metadata.version(pkg)
    except importlib.metadata.PackageNotFoundError:
        env[pkg] = None
add_table("environment", pd.DataFrame([{"item": k, "value": v} for k, v in env.items()]),
          "Software versions and device used for this analysis run.")

readme = pd.DataFrame([{"sheet": name, "rows": len(df), "description": desc} for name, (desc, df) in TABLES.items()])
readme = pd.concat([pd.DataFrame([{"sheet": "(run)", "rows": None,
                                   "description": f"{RUN_ID} | created {datetime.datetime.now(datetime.timezone.utc).isoformat()}"
                                                  f" | settings changed from reference: {list(CONFIG_CHANGES) or 'none'}"}]),
                    readme], ignore_index=True)

out_path = os.path.join(RUN_DIR, "results_all_tables.xlsx")
with pd.ExcelWriter(out_path, engine="openpyxl") as xl:
    readme.to_excel(xl, sheet_name="README", index=False)
    for name, (desc, df) in TABLES.items():
        df.to_excel(xl, sheet_name=name, index=False)
print(f"saved {len(TABLES)} tables -> {out_path}")

In [ ]:
# blinded list of the disagreements for the consensus meeting (no method or cluster information)
dis = m[m["r1"] != m["r2"]][["pair_id"]].merge(
    pd.read_csv(key_path)[["pair_id", "response_a", "response_b"]], on="pair_id")
dis = dis.merge(m[["pair_id", "r1", "r2"]], on="pair_id").rename(
    columns={"r1": "reviewer1", "r2": "reviewer2"}).assign(consensus="", reason="")
dis_path = os.path.join(VALIDATION_DIR, "consensus_meeting.xlsx")
dis.to_excel(dis_path, index=False)
print(f"{len(dis)} disagreements -> {dis_path}")

In [ ]:
# GROUP SIZES: per prompt, how many responses fall in each semantic group
# Adds a 'group_sizes' sheet to results_all_tables.xlsx and saves a CSV copy.

def sizes(s):
    """Group sizes, largest first, e.g. '8 + 2 + 1'."""
    return " + ".join(str(c) for c in s.value_counts().values)

v = calls[calls["valid"]]
group_sizes = (v.groupby(["model", "key"], sort=False)
                .apply(lambda g: pd.Series({
                    "n_responses": len(g),
                    "labels": sizes(g["parsed_label"]),
                    "n_groups_baseline": g["cluster_baseline"].nunique(),
                    "groups_baseline": sizes(g["cluster_baseline"]),
                    "n_groups_modified": g["cluster_modified"].nunique(),
                    "groups_modified": sizes(g["cluster_modified"])}))
                .reset_index())

# show the table
print(group_sizes.to_string(index=False))

# save a CSV copy in the run folder
group_sizes.to_csv(os.path.join(RUN_DIR, "semantic_group_sizes.csv"), index=False)

xlsx_path = os.path.join(RUN_DIR, "results_all_tables.xlsx")
with pd.ExcelWriter(xlsx_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as xl:
    group_sizes.to_excel(xl, sheet_name="group_sizes", index=False)
print(f"saved {len(group_sizes)} prompts -> 'group_sizes' sheet in {xlsx_path}")